In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

if "notebooks" in os.getcwd():
    os.chdir(Path.cwd().parent)

In [ ]:
import logging

logger = logging.getLogger("ts_visualization")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(levelname)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [ ]:
import numpy as np
from lets_plot import LetsPlot

from xaitimesynth import (
    TimeSeriesBuilder,
    constant,
    gaussian,
    list_components,
    list_feature_components,
    list_signal_components,
    manual,
    peak,
    plot_component,
    plot_components,
    random_walk,
    seasonal,
    trend,
)
from xaitimesynth.generators import generate_ecg_like, generate_random_walk

# Enable lets_plot for notebook display
LetsPlot.setup_html()

# Helper functions

# Synth Data

In [ ]:
# see generators/components already defined in package
print(list_components())
print(list_signal_components())
print(list_feature_components())

## Single components

In [ ]:
signal = generate_random_walk(1000, np.random.RandomState(42), step_size=0.1)
plot_component(signal)

In [ ]:
plot_component(component_type="red_noise", n_timesteps=1000, phi=0.99)

In [ ]:
plot_component(generate_ecg_like(400, np.random.RandomState(42))).show()
plot_component(component_type="ecg_like", n_timesteps=400, line_color="blue").show()

In [ ]:
# Generate and plot a random walk
plot_component(component_type="random_walk", n_timesteps=100, step_size=0.2).show()

# Generate and plot a seasonal pattern
plot_component(
    component_type="seasonal", n_timesteps=200, period=20, amplitude=1.5
).show()

# Example 1: Using a manual component with custom values

# Create custom values for a zigzag pattern
zigzag = np.concatenate([np.linspace(0, 1, 10), np.linspace(1, 0, 10)])
zigzag = np.tile(zigzag, 5)  # Repeat the pattern 5 times

# Plot using the manual component with custom values
plot_component(
    component_type="manual",
    values=zigzag,
    n_timesteps=100,
    line_color="red",
).show()


# Define a custom generator function for a damped oscillation
def damped_oscillation(length, rng, frequency=0.1, decay=0.05, **kwargs):
    t = np.arange(length)
    return np.exp(-decay * t) * np.sin(2 * np.pi * frequency * t)


# Plot using the manual component with custom generator
plot_component(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
).show()

plot_component(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
    normalization_kwargs={"feature_range": (-1, 1)},  # Custom normalization range
).show()

## Univariate time series

In [ ]:
sine_levelshift_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)
    .add_signal(seasonal(period=5), role="foundation")
    .add_feature(seasonal(period=5), start_pct=0.2, end_pct=0.4)
    .add_feature(constant(value=2.0), start_pct=0.5, end_pct=0.7)
    .for_class(1)
    .add_signal(gaussian(), role="foundation")
    .add_feature(constant(value=2.0), start_pct=0.5, end_pct=0.7)
    .build()
)
# print(sine_levelshift_dataset.keys())

plot_components(sine_levelshift_dataset).show()

In [ ]:
# Cylinder-Bell-Funnel-like dataset
gauss_sigma = 1
dataset_builder = (
    TimeSeriesBuilder(
        n_timesteps=128, n_samples=200, random_state=42, normalization="none"
    )
    # cylinder
    .for_class(0)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(constant(value=6), length_pct=0.25, random_location=True)
    # bell
    .for_class(1)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(trend(endpoints=[0, 6]), length_pct=0.25, random_location=True)
    # funnel
    .for_class(2)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(trend(endpoints=[6, 0]), length_pct=0.25, random_location=True)
)
dataset = dataset_builder.build()
dataset
plot_components(dataset_builder.build()).show()

In [ ]:
# Create a custom component and use it
def my_seasonal_pattern(n_timesteps, rng, frequency=0.1, amplitude=1.0):
    """Generate a custom seasonal pattern."""
    t = np.arange(n_timesteps)
    return amplitude * np.sin(2 * np.pi * frequency * t / n_timesteps)


custom_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=10)
    .for_class(0)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)

plot_components(custom_dataset).show()

## Multivariate time series

In [ ]:
# Create a 3-dimensional time series dataset
multivariate_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, n_dimensions=3, random_state=42)
    # Class 0: Random walk in all dimensions but no discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])  # Apply to all dimensions
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Class 1: Features in different dimensions
    .for_class(1)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Add shapelet only to dimensions 0 and 1 with shared location (same position in both)
    .add_feature(
        constant(value=1.2),
        random_location=True,
        length_pct=0.15,
        dim=[0, 1],
        shared_location=True,
    )
    # Add peak only to dimension 2
    .add_feature(
        peak(amplitude=1.0, width=1),
        random_location=True,
        length_pct=0.05,
        dim=[0, 1, 2],
    )
    # Add level change to all dimensions, with different locations for each
    # .add_feature(
    #     level_change(amplitude=0.7),
    #     random_location=True,
    #     length_pct=0.2,
    #     dim=[0, 1, 2],
    #     shared_location=False,
    # )
    .build()
)

print(f"Dataset shape: {multivariate_dataset['X'].shape}")  # Should be (50, 100, 3)
print(f"Number of dimensions: {multivariate_dataset['metadata']['n_dimensions']}")

# Example of accessing a specific dimension
dim0_data = multivariate_dataset["X"][:, :, 0]  # First dimension for all samples
print(f"Dimension 0 shape: {dim0_data.shape}")

# Convert to DataFrame with specific dimensions
df = TimeSeriesBuilder().to_df(multivariate_dataset, dimensions=[0, 1])
print(df.head())

plot_components(multivariate_dataset)

In [ ]:
# For a single plot or list with one element
viz = plot_components(multivariate_dataset, dimensions=[0])
# viz.show()

# For multiple plots
vizs = plot_components(multivariate_dataset, dimensions=[0, 1])
for i, viz in enumerate(vizs):
    print(f"Dimension {i}:")
    viz.show()

## Creating Train/Test/Validation Splits with clone()

The clone method allows creating multiple datasets from the same builder with different parameters.

In [ ]:
# Create a base builder with class definitions
base_builder = (
    TimeSeriesBuilder(n_timesteps=100, random_state=42)
    .for_class(0)  # Class 0: Just random walk with noise
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk with noise plus features
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(constant(value=1.0), start_pct=0.4, end_pct=0.6)
)

train_dataset = base_builder.clone(n_samples=140, random_state=42).build()
test_dataset = base_builder.clone(n_samples=40, random_state=43).build()
val_dataset = base_builder.clone(n_samples=20, random_state=44).build()

print(
    f"Train dataset: X shape={train_dataset['X'].shape}, y shape={train_dataset['y'].shape}"
)
print(
    f"Test dataset: X shape={test_dataset['X'].shape}, y shape={test_dataset['y'].shape}"
)
print(
    f"Validation dataset: X shape={val_dataset['X'].shape}, y shape={val_dataset['y'].shape}"
)

# Compare with explainer indexes
# This is now much simpler since each dataset has its own indexing starting from 0
print("\nSample index 2 in training set is truly sample index 2 (not an offset)")
print("Sample index 2 in test set is truly sample index 2 (not an offset)")

## Metrics

### Correlation Score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from xaitimesynth.builder import TimeSeriesBuilder
from xaitimesynth.metrics import correlation_score

# Create a simple dataset with seasonal features
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=2, random_state=42)
    .for_class(0)
    .add_signal(gaussian(mu=0, sigma=0.1), role="foundation")
    .add_feature(seasonal(period=10, amplitude=2.0), start_pct=0.3, end_pct=0.7)
    .build(return_components=True)
)

# Generate different attribution examples
n_timesteps = dataset["metadata"]["n_timesteps"]
n_dimensions = dataset["metadata"]["n_dimensions"]

# Perfect attribution (exactly matches the seasonal feature)
perfect_attr = np.zeros((2, n_timesteps, n_dimensions))
for sample_idx in range(2):
    # Extract the feature component for each sample
    seasonal_feature = None
    for feature_name, feature_values in dataset["components"][
        sample_idx
    ].features.items():
        if "seasonal" in feature_name:
            seasonal_feature = feature_values
            break

    # Copy feature values to attribution (only where features exist - not NaN)
    for t in range(n_timesteps):
        if not np.isnan(seasonal_feature[t]):
            perfect_attr[sample_idx, t, 0] = seasonal_feature[t]

# Inversely correlated attribution
inverse_attr = -1 * perfect_attr

# Random attribution (no correlation with features)
random_attr = np.random.RandomState(42).randn(2, n_timesteps, n_dimensions)

# Calculate correlation scores
absolute_corr_perfect = correlation_score(
    perfect_attr, dataset, feature_source="isolated", absolute=True
)
raw_corr_perfect = correlation_score(
    perfect_attr, dataset, feature_source="isolated", absolute=False
)
absolute_corr_inverse = correlation_score(
    inverse_attr, dataset, feature_source="isolated", absolute=True
)
raw_corr_inverse = correlation_score(
    inverse_attr, dataset, feature_source="isolated", absolute=False
)
random_corr = correlation_score(
    random_attr, dataset, feature_source="isolated", absolute=True
)

# Print the results
print(f"Perfect attribution absolute correlation: {absolute_corr_perfect:.4f}")
print(f"Perfect attribution raw correlation: {raw_corr_perfect:.4f}")
print(f"Inverse attribution absolute correlation: {absolute_corr_inverse:.4f}")
print(f"Inverse attribution raw correlation: {raw_corr_inverse:.4f}")
print(f"Random attribution correlation: {random_corr:.4f}")

# Plot for visual comparison
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
sample_idx = 0

# Extract feature mask
feature_key = next(k for k in dataset["feature_masks"].keys())
mask = dataset["feature_masks"][feature_key][sample_idx]

# Extract original time series
X = dataset["X"][sample_idx, :, 0]

# Extract feature component
seasonal_feature = np.full(n_timesteps, np.nan)
for feature_name, feature_values in dataset["components"][sample_idx].features.items():
    if "seasonal" in feature_name:
        for t in range(n_timesteps):
            if not np.isnan(feature_values[t]):
                seasonal_feature[t] = feature_values[t]

# Plot 1: Original time series with highlighted feature regions
axes[0, 0].plot(X, label="Time Series")
axes[0, 0].fill_between(
    np.arange(n_timesteps),
    np.min(X),
    np.max(X),
    where=mask,
    alpha=0.3,
    color="green",
    label="Feature Region",
)
axes[0, 0].set_title("Time Series with Feature Regions")
axes[0, 0].legend()

# Plot 2: Isolated feature component
axes[0, 1].plot(np.arange(n_timesteps), seasonal_feature, label="Seasonal Feature")
axes[0, 1].set_title("Isolated Feature Component")
axes[0, 1].legend()

# Plot 3: Perfect attribution
axes[1, 0].plot(perfect_attr[sample_idx, :, 0], label="Perfect Attribution")
axes[1, 0].plot(seasonal_feature, "r--", alpha=0.5, label="Feature")
axes[1, 0].set_title(f"Perfect Attribution (corr={raw_corr_perfect:.4f})")
axes[1, 0].legend()

# Plot 4: Inverse attribution
axes[1, 1].plot(inverse_attr[sample_idx, :, 0], label="Inverse Attribution")
axes[1, 1].plot(seasonal_feature, "r--", alpha=0.5, label="Feature")
axes[1, 1].set_title(f"Inverse Attribution (corr={raw_corr_inverse:.4f})")
axes[1, 1].legend()

# Plot 5: Random attribution
axes[2, 0].plot(random_attr[sample_idx, :, 0], label="Random Attribution")
axes[2, 0].plot(seasonal_feature, "r--", alpha=0.5, label="Feature")
axes[2, 0].set_title(f"Random Attribution (corr={random_corr:.4f})")
axes[2, 0].legend()

# Plot 6: Scatter plot of attribution vs feature (in feature regions only)
valid_indices = ~np.isnan(seasonal_feature)
axes[2, 1].scatter(
    seasonal_feature[valid_indices],
    perfect_attr[sample_idx, valid_indices, 0],
    label="Perfect",
    alpha=0.6,
)
axes[2, 1].scatter(
    seasonal_feature[valid_indices],
    inverse_attr[sample_idx, valid_indices, 0],
    label="Inverse",
    alpha=0.6,
)
axes[2, 1].axline([0, 0], [1, 1], linestyle="--", color="gray", label="y=x")
axes[2, 1].axline([0, 0], [1, -1], linestyle="--", color="red", label="y=-x")
axes[2, 1].set_xlabel("Feature Value")
axes[2, 1].set_ylabel("Attribution Value")
axes[2, 1].set_title("Attribution vs. Feature Values")
axes[2, 1].legend()

plt.tight_layout()
plt.show()